In [187]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import balanced_accuracy_score
from sklearn.feature_selection import SelectKBest, mutual_info_classif, SequentialFeatureSelector
from functools import partial
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import random as random

# Please do not change these random seeds.
np.random.seed(231)
random.seed(231)
model = KNeighborsClassifier(n_neighbors=5)

In [188]:
# --------------------------- Part 1: Load the Data ---------------------------

data_train = pd.read_csv('Training.csv')
data_train = data_train.rename(columns={'Tryglicerides': 'Triglycerides'})
X_train, y_train = data_train.drop('Status', axis=1), data_train['Status']

data_test = pd.read_csv('Test.csv')
data_test = data_test.rename(columns={'Tryglicerides': 'Triglycerides'})
X_test, y_test = data_test.drop('Status', axis=1), data_test['Status']

print(X_train.head(5))

              Drug    Age Sex Ascites Hepatomegaly Spiders Edema  Bilirubin  \
0  D-penicillamine  19567   F       N            Y       N     S        2.3   
1          Placebo  17246   F       N            Y       N     N        2.1   
2          Placebo  17874   F       N            Y       N     S        8.7   
3  D-penicillamine  15895   F       Y            Y       Y     S       17.1   
4          Placebo  24650   F       N            Y       N     N        8.0   

   Cholesterol  Albumin  Copper  Alk_Phos    SGOT  Triglycerides  Platelets  \
0        260.0     3.18   231.0   11320.2  105.78           94.0      216.0   
1        262.0     3.48    58.0    2045.0   89.90           84.0      225.0   
2        310.0     3.89   107.0     637.0  117.00          242.0      298.0   
3        674.0     2.53   207.0    2078.0  182.90          598.0      268.0   
4        468.0     2.81   139.0    2009.0  198.40          139.0      233.0   

   Prothrombin  Stage  
0         12.4    3.0  
1 

In [189]:
# Hard coded for funsies, 
# Previous iteration was 
#     for col in categorical_cols:
#        df[col] = df[col].astype('category').cat.codes.replace(-1, np.nan)
# in imputeMissingValues(), but may result in unpredictable behaviour

def encodeValues(df):
    from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
    
    encoder = OneHotEncoder()
    encoded = encoder.fit_transform(df[['Drug', 'Sex', 'Ascites', 'Hepatomegaly', 'Spiders', 'Edema']])
    
    stage_order = ['1', '2', '3', '4']
    encoder = OrdinalEncoder(categories=[stage_order])
    encoded = encoder.fit_transform(df[['Stage']])

    return encoded

In [190]:
def imputeMissingValues(df, numerical_cols, categorical_cols):
    from sklearn.experimental import enable_iterative_imputer
    from sklearn.impute import IterativeImputer
    from sklearn.ensemble import RandomForestRegressor

    num_imputer = IterativeImputer(
        estimator=RandomForestRegressor(n_estimators=50, random_state=42),
        max_iter = 10,
        random_state = 42
    )

    df[numerical_cols] = num_imputer.fit_transform(df[numerical_cols])

    for col in categorical_cols:
        df[col] = df[col].fillna(df[col].mode()[0])  
        
    return df


In [ ]:
# --------------------------- Part 1: Data Preprocessing ---------------------------

# Identify numerical and categorical columns
from sklearn.preprocessing import MinMaxScaler, StandardScaler

numerical_cols = ['Age', 'Bilirubin', 'Cholesterol', 'Albumin', 'Copper', 'Alk_Phos', 'SGOT', 'Triglycerides', 'Platelets', 'Prothrombin']  # List of numerical feature names
categorical_cols = ['Drug', 'Sex', 'Ascites', 'Hepatomegaly', 'Spiders', 'Edema', 'Stage'] # List of categorical feature names

### Step 1: Handle Missing Values ###
# Define and apply appropriate imputers for numerical and categorical features.
# Ensure consistency between X_train and X_test.

X_train = imputeMissingValues(X_train, numerical_cols, categorical_cols)
X_test = imputeMissingValues(X_test, numerical_cols, categorical_cols)

print(X_train.head(5))
### Step 2: Encoding Categorical Features ###
# Convert categorical features into numerical format using encoding techniques.

# X_train = pd.DataFrame(encodeValues(X_train))
# X_test = pd.DataFrame(encodeValues(X_test))
from sklearn.impute import SimpleImputer
# Impute numerical features with median

from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
    
encoder = OneHotEncoder(sparse_output=False)
encoder.set_output(transform="pandas")
X_train_encoded = encoder.fit_transform(X_train[['Drug', 'Sex', 'Ascites', 'Hepatomegaly', 'Spiders', 'Edema']])
    
stage_order = ['1', '2', '3', '4']
encoder = OrdinalEncoder(categories=[stage_order])
encoder.set_output(transform="pandas")
X_train_encoded = encoder.fit_transform(X_train[['Stage']])

print(X_train_encoded.head(5))



# Checked for any strong correlations between categorical features using crosstab, but did not find any strong correlations that would warrant dropping any features.
# for col1 in categorical_cols: 
#    for col2 in categorical_cols:
#        print(pd.crosstab(X_train[col1], X_train[col2]))

### Step 3: Feature Scaling ###
# Apply scaling/normalization to features.



              Drug      Age Sex Ascites Hepatomegaly Spiders Edema  Bilirubin  \
0  D-penicillamine  19567.0   F       N            Y       N     S        2.3   
1          Placebo  17246.0   F       N            Y       N     N        2.1   
2          Placebo  17874.0   F       N            Y       N     S        8.7   
3  D-penicillamine  15895.0   F       Y            Y       Y     S       17.1   
4          Placebo  24650.0   F       N            Y       N     N        8.0   

   Cholesterol  Albumin  Copper  Alk_Phos    SGOT  Triglycerides  Platelets  \
0        260.0     3.18   231.0   11320.2  105.78           94.0      216.0   
1        262.0     3.48    58.0    2045.0   89.90           84.0      225.0   
2        310.0     3.89   107.0     637.0  117.00          242.0      298.0   
3        674.0     2.53   207.0    2078.0  182.90          598.0      268.0   
4        468.0     2.81   139.0    2009.0  198.40          139.0      233.0   

   Prothrombin  Stage  
0         12.4

In [ ]:
# --------------------------- Part 2: Feature Ranking ---------------------------
# Use SelectKBest with mutual_info_classif to select the top 7 features from the processed training set X_train_process.
# You should use mutual_info_fix as the parameter for SelectKBest to ensure random_state is set to 231 for reproducibility.
mutual_info_fix = partial(mutual_info_classif, random_state=231)
selector_kbest = SelectKBest(score_func=mutual_info_fix, k=7)
X_train_kbest = selector_kbest.fit_transform(X_train_process, y_train)
X_test_kbest = selector_kbest.transform(X_test_process)

# Train the model with selected features.
model.fit(X_train_kbest, y_train)
y_pred = model.predict(X_test_kbest)
print(f"Feature Ranking: {balanced_accuracy_score(y_test, y_pred):.4f}")

In [ ]:
# --------------------------- Part 3: Sequential Feature Selection ---------------------------
# Use Sequential Backward Selection (SBS) to select a subset of 7 features from the processed training set X_train_process.
selector_sequential = []  # Replace [] with SequentialFeatureSelector implementation
X_train_sbfs = []  # Replace [] with transformed training data
X_test_sbfs = []  # Replace [] with transformed test data

# Train the model with selected features.
model.fit(X_train_sbfs, y_train)
y_pred = model.predict(X_test_sbfs)
print(f"Sequential Feature Selection: {balanced_accuracy_score(y_test, y_pred):.4f}")